In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu

np.random.seed(42)
import random
random.seed(42)

In [2]:
mdata = mu.read("./data/EAE/CCA/EAE_HV_then_merged_test293883_singlets_CVscores.h5mu")
mdata

MuData object with n_obs × n_vars = 40484 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length'
  uns:	'cca_x_weights', 'cca_y_weights'
  obsm:	'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call', 'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_atchley_pairwise', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_atchley_pairwise', 'X_VJ_1_cdr3_aa_composition', 'tcr_embs'
  2 modalities
    airr:	40484 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'
    gex:	40484 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group', 'CV_score_0', 'CV_score_1', 'CV_score_2', 'CV_score_0_high', 'CV_score_0_high_cat', 'CV_score_1_high', 'CV_score_1_high_cat', 'CV_score_2_high', 'CV_score_2_high_cat', 'CNS_score', 'SPL_score'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'Tissue_group_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'rank_genes_groups', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

In [3]:
tcr_embs = pd.DataFrame(mdata.obsm['tcr_embs'], index=mdata.obs_names)
tcr_embs.head(5)

,0,1,2,3,4,5,6,7,8,9,...,1371,1372,1373,1374,1375,1376,1377,1378,1379,1380
AAGTAGCAGATAGGCG-1_0516_CNS,0.0,0.0,0.314295,-0.338025,0.037813,0.319511,-0.072471,0.300712,0.378219,-0.418578,...,-0.020496,-0.049762,-0.020496,-0.066642,-0.102511,-0.12317,0.0,-1.207124,0.965137,0.0
AAGTATACACCCAGTA-1_0516_CNS,0.0,0.0,0.314295,-0.338025,0.037813,0.319511,-0.072471,0.300712,0.378219,-0.418578,...,-0.020496,-0.049762,-0.020496,-0.066642,-0.102511,-0.12317,0.0,0.694143,-0.036211,0.0
AAGTTTGGTGGAACCC-1_0516_CNS,0.0,0.0,0.314295,-0.338025,0.037813,0.319511,-0.072471,0.300712,0.378219,-0.418578,...,-0.020496,-0.049762,-0.020496,-0.066642,-0.102511,-0.12317,0.0,1.644776,-0.036211,0.0
AATTCGCCAGCTGTAT-1_0516_CNS,0.0,0.0,0.314295,-0.338025,0.037813,0.319511,-0.072471,0.300712,0.378219,-0.418578,...,-0.020496,-0.049762,-0.020496,-0.066642,-0.102511,-0.12317,0.0,0.694143,-1.037559,0.0
AGCAAGTAGAAATTGG-1_0516_CNS,0.0,0.0,0.314295,-0.338025,0.037813,0.319511,-0.072471,0.300712,0.378219,-0.418578,...,-0.020496,-0.049762,-0.020496,-0.066642,-0.102511,-0.12317,0.0,1.644776,0.965137,0.0


In [4]:
gex_df = mdata['gex'].to_df()
gex_df = gex_df.loc[:, ~gex_df.columns.str.startswith('mt-')]
gex_df.head(5)

,1110034G24Rik,1110037F02Rik,1500009L16Rik,1700003C15Rik,1700012B07Rik,1700016L21Rik,1700019D03Rik,1700025G04Rik,1700028E10Rik,1700061F12Rik,...,Zfp831,Zfyve28,Zg16,Zhx2,Zmym2,Zmym4,Znrf3,Zswim6,Zup1,Zzz3
AAGTAGCAGATAGGCG-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0
AAGTATACACCCAGTA-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,2.228830,0.000000,0.0
AAGTTTGGTGGAACCC-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0
AATTCGCCAGCTGTAT-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.201334,0.0,0.0,1.613137,1.613137,0.0
AGCAAGTAGAAATTGG-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0


In [19]:
labels = mdata['gex'].obs[['tissue', 'cell_type', 'state', 'GSE', 'sample_id', 'CV_score_0', 'CV_score_1', 'CV_score_2']]
labels = pd.concat([labels, mdata.obs['set']], axis=1)
labels = pd.concat([labels, mdata['airr'].obs['clone_id_size']], axis=1)
labels.columns = [f"label_{col}" for col in labels.columns]

labels.head(5)

,label_tissue,label_cell_type,label_state,label_GSE,label_sample_id,label_CV_score_0,label_CV_score_1,label_CV_score_2,label_set,label_clone_id_size
AAGTAGCAGATAGGCG-1_0516_CNS,CNS,CD4,Activation,LEE,5_7,0.049247,1.154527,0.485142,train,1.0
AAGTATACACCCAGTA-1_0516_CNS,CNS,NaN,Activation,LEE,5_7,-0.172905,-1.429184,-0.273997,train,1.0
AAGTTTGGTGGAACCC-1_0516_CNS,CNS,NaN,Exhaust,LEE,5_3,0.010912,0.491048,-0.117987,train,1.0
AATTCGCCAGCTGTAT-1_0516_CNS,CNS,Treg,Activation,LEE,5_6,-0.074756,-0.415768,0.490410,train,1.0
AGCAAGTAGAAATTGG-1_0516_CNS,CNS,Treg,Exhaust,LEE,5_3,-0.230434,0.223790,1.063392,train,1.0


In [7]:
labels['label_set'].value_counts()

label_set
train    34952
test      5532
Name: count, dtype: int64

### Save Raw gex and tcr

In [20]:
import os
output_cd = 'raw_data'
if not os.path.exists(output_cd):
    os.makedirs(output_cd)
    tcr_embs.to_csv(os.path.join(output_cd, "tcr_embs.csv"), index=True)
    gex_df.to_csv(os.path.join(output_cd, "gex_df.csv"), index=True)
    labels.to_csv(os.path.join(output_cd, "labels.csv"), index=True)
    
else:
    print(f"Directory {output_cd} already exists")




Directory raw_data already exists


### Save cca transformed gex and tcr

In [15]:
# selet weight with highest corr along target label
cv_dim = 2

gex_cca = mdata['gex'].to_df().to_numpy()*mdata.uns['cca_x_weights'][:, cv_dim]
tcr_cca = mdata.obsm['tcr_embs']*mdata.uns['cca_y_weights'][:, cv_dim]

print(gex_cca.shape)
print(tcr_cca.shape)

(40484, 3001)
(40484, 1381)


In [ ]:
gex_cca_df = pd.DataFrame(gex_cca, index=mdata.obs_names)
tcr_cca_df = pd.DataFrame(tcr_cca, index=mdata.obs_names)

output_cd = 'cca_data'
if not os.path.exists(output_cd):
    os.makedirs(output_cd)
    tcr_cca_df.to_csv(os.path.join(output_cd, "tcr_embs.csv"), index=True)
    gex_cca_df.to_csv(os.path.join(output_cd, "gex_df.csv"), index=True)
    labels.to_csv(os.path.join(output_cd, "labels.csv"), index=True)
else:
    print(f"Directory {output_cd} already exists")

###  Raw data + CCA scores